# Black–Scholes Simulation + Neural-Network Trading Agent

**A self-contained notebook for Google Colab.**

This notebook does four things:

1. **Simulates** the price dynamics that underpin the **Black–Scholes (BS)** model
   (Geometric Brownian Motion) and implements the closed-form BS option pricer and
   its *Greeks*.
2. **Builds a neural network** that learns a **trading decision policy**
   (go long / stay flat / go short) from features engineered around the BS /
   volatility framework.
3. **Backtests** the resulting algorithm on **real historical market data**.
4. **Reports the results as charts** (equity curve, drawdown, Sharpe, signal
   distribution, etc.).

Everything is written in English and the relevant **theory is included inline**.

---

### Table of contents
1. [Setup & imports](#setup)
2. [Part I — Black–Scholes theory](#bs-theory)
3. [Part I — Geometric Brownian Motion simulation](#gbm)
4. [Part I — Black–Scholes pricing & Greeks](#bs-pricing)
5. [Part II — Neural-network trading: theory](#nn-theory)
6. [Part II — Real historical data](#data)
7. [Part II — Feature engineering (BS / volatility based)](#features)
7b. [Part II — Strategy configuration (the levers)](#config)
8. [Part II — Building the datasets: tabular & sequences](#datasets)
9. [Part II — A zoo of prediction models](#zoo)
10. [Part III — Backtesting all models](#backtest)
11. [Part III — Results & charts](#results)
12. [Conclusions & caveats](#conclusions)

> ⚠️ **Disclaimer.** This notebook is for **education and research** only. It is
> *not* financial advice. Backtested performance does not guarantee future
> results, and the model deliberately keeps things simple for clarity.


<a id="setup"></a>
## 1. Setup & imports

Run this cell first. In Colab, `numpy`, `pandas`, `matplotlib`, `scipy`,
`scikit-learn` and `tensorflow` are already installed; we only need to add
`yfinance` for downloading real market data.


In [ ]:
# Install the one dependency Colab may be missing.
# (Safe to re-run; it is a no-op if already installed.)
import sys, subprocess
try:
    import yfinance  # noqa: F401
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
np.random.seed(SEED)

# TensorFlow / Keras
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("NumPy   :", np.__version__)
print("Pandas  :", pd.__version__)
print("TF/Keras:", tf.__version__)


<a id="bs-theory"></a>
## 2. Part I — Black–Scholes theory

### 2.1 The model of the underlying

The Black–Scholes framework assumes the price of the underlying asset
$S_t$ follows a **Geometric Brownian Motion (GBM)**:

$$
dS_t = \mu\, S_t\, dt + \sigma\, S_t\, dW_t,
$$

where

- $\mu$ is the (real-world) **drift** / expected return,
- $\sigma$ is the **volatility**,
- $W_t$ is a standard **Brownian motion** (Wiener process).

Applying Itô's lemma to $\ln S_t$ gives the closed-form solution

$$
S_t = S_0 \, \exp\!\Big[\big(\mu - \tfrac12\sigma^2\big)t + \sigma W_t\Big],
$$

so **log-returns are normally distributed** and prices are **log-normal**.

### 2.2 The Black–Scholes PDE

Under a no-arbitrage argument with continuous **delta-hedging**, the value
$V(S,t)$ of any European derivative satisfies the **Black–Scholes PDE**:

$$
\frac{\partial V}{\partial t}
+ \tfrac12\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}
+ r S \frac{\partial V}{\partial S}
- r V = 0,
$$

where $r$ is the **risk-free rate**. Note the drift $\mu$ disappears: pricing is
done under the **risk-neutral measure** where the asset drifts at $r$.

### 2.3 Closed-form European option prices

For a European **call** ($C$) and **put** ($P$) with strike $K$ and time to
maturity $T$:

$$
d_1 = \frac{\ln(S/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}},
\qquad
d_2 = d_1 - \sigma\sqrt{T},
$$

$$
C = S\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2),
\qquad
P = K e^{-rT}\,\Phi(-d_2) - S\,\Phi(-d_1),
$$

where $\Phi$ is the standard normal CDF. Put–call parity holds:
$C - P = S - K e^{-rT}$.

### 2.4 The Greeks

The **Greeks** are sensitivities of the option value used for hedging and risk.
For a call:

| Greek | Meaning | Formula |
|-------|---------|---------|
| $\Delta$ | $\partial V/\partial S$ | $\Phi(d_1)$ |
| $\Gamma$ | $\partial^2 V/\partial S^2$ | $\dfrac{\phi(d_1)}{S\sigma\sqrt{T}}$ |
| $\mathcal{V}$ (Vega) | $\partial V/\partial \sigma$ | $S\,\phi(d_1)\sqrt{T}$ |
| $\Theta$ | $\partial V/\partial t$ | $-\dfrac{S\phi(d_1)\sigma}{2\sqrt{T}} - rKe^{-rT}\Phi(d_2)$ |
| $\rho$ | $\partial V/\partial r$ | $KTe^{-rT}\Phi(d_2)$ |

$\phi$ is the standard normal PDF. **Delta** is the key link to trading: it is the
number of units of the underlying needed to hedge the option, and — as we'll use
later — a natural, bounded way to translate a *directional forecast* into a
*position size*.


<a id="gbm"></a>
## 3. Part I — Simulating Geometric Brownian Motion

We now simulate GBM paths (the "world" the BS model assumes) and check that the
empirical distribution of log-returns matches the theory.


In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, seed=None):
    '''Simulate Geometric Brownian Motion paths.

    dS = mu*S*dt + sigma*S*dW  ->  exact log-Euler scheme.

    Returns
    -------
    t     : (n_steps+1,) time grid
    paths : (n_paths, n_steps+1) simulated price paths
    '''
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    # Brownian increments
    dW = rng.normal(0.0, np.sqrt(dt), size=(n_paths, n_steps))
    # Log-return increments (exact solution of GBM)
    incr = (mu - 0.5 * sigma**2) * dt + sigma * dW
    log_paths = np.concatenate(
        [np.zeros((n_paths, 1)), np.cumsum(incr, axis=1)], axis=1
    )
    paths = S0 * np.exp(log_paths)
    t = np.linspace(0.0, T, n_steps + 1)
    return t, paths


# Parameters
S0, mu, sigma, T = 100.0, 0.08, 0.20, 1.0
n_steps, n_paths = 252, 200

t, paths = simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, seed=SEED)
print("Simulated", paths.shape[0], "paths over", paths.shape[1], "time points.")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# (a) A sample of simulated paths
for i in range(40):
    ax[0].plot(t, paths[i], lw=0.8, alpha=0.6)
ax[0].plot(t, S0 * np.exp(mu * t), color="black", lw=2.5, label=r"$E[S_t]=S_0e^{\mu t}$")
ax[0].set_title("Simulated GBM price paths")
ax[0].set_xlabel("Time (years)"); ax[0].set_ylabel("Price"); ax[0].legend()

# (b) Terminal log-returns vs the theoretical normal density
terminal_log_ret = np.log(paths[:, -1] / S0)
ax[1].hist(terminal_log_ret, bins=30, density=True, alpha=0.6, label="Simulated")
xs = np.linspace(terminal_log_ret.min(), terminal_log_ret.max(), 200)
theo = norm.pdf(xs, (mu - 0.5 * sigma**2) * T, sigma * np.sqrt(T))
ax[1].plot(xs, theo, "r-", lw=2, label="Theoretical N")
ax[1].set_title("Terminal log-returns vs Black–Scholes theory")
ax[1].set_xlabel(r"$\ln(S_T/S_0)$"); ax[1].set_ylabel("Density"); ax[1].legend()

plt.tight_layout(); plt.show()


<a id="bs-pricing"></a>
## 4. Part I — Black–Scholes pricing & Greeks

We implement the closed-form pricer and the Greeks, then visualise how the call
price and its Delta behave across the moneyness spectrum.


In [ ]:
def bs_price(S, K, T, r, sigma, option="call"):
    '''Black–Scholes price of a European option (vectorised).'''
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    T = np.maximum(T, 1e-12); sigma = np.maximum(sigma, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def bs_greeks(S, K, T, r, sigma, option="call"):
    '''Return a dict with Delta, Gamma, Vega, Theta, Rho.'''
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    T = np.maximum(T, 1e-12); sigma = np.maximum(sigma, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf = norm.pdf(d1)
    if option == "call":
        delta = norm.cdf(d1)
        theta = (-S * pdf * sigma / (2 * np.sqrt(T))
                 - r * K * np.exp(-r * T) * norm.cdf(d2))
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
    else:
        delta = norm.cdf(d1) - 1.0
        theta = (-S * pdf * sigma / (2 * np.sqrt(T))
                 + r * K * np.exp(-r * T) * norm.cdf(-d2))
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
    gamma = pdf / (S * sigma * np.sqrt(T))
    vega = S * pdf * np.sqrt(T)
    return {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta, "rho": rho}


# Sanity check: put-call parity  C - P = S - K e^{-rT}
K, r = 100.0, 0.03
C = bs_price(100, K, 1.0, r, 0.2, "call")
P = bs_price(100, K, 1.0, r, 0.2, "put")
print(f"Call = {C:.4f}, Put = {P:.4f}")
print(f"C - P = {C - P:.4f}  vs  S - K e^-rT = {100 - K*np.exp(-r*1.0):.4f}")


In [ ]:
S_grid = np.linspace(60, 140, 200)
call_prices = bs_price(S_grid, K, T=0.5, r=r, sigma=0.2, option="call")
greeks = bs_greeks(S_grid, K, T=0.5, r=r, sigma=0.2, option="call")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(S_grid, call_prices, lw=2)
ax[0].axvline(K, color="gray", ls="--", label="Strike K")
ax[0].plot(S_grid, np.maximum(S_grid - K, 0), "k:", label="Payoff at maturity")
ax[0].set_title("Black–Scholes call price"); ax[0].set_xlabel("Spot S")
ax[0].set_ylabel("Option value"); ax[0].legend()

ax[1].plot(S_grid, greeks["delta"], lw=2, label=r"$\Delta$ (call)")
ax[1].plot(S_grid, greeks["gamma"] * 10, lw=2, label=r"$\Gamma \times 10$")
ax[1].axvline(K, color="gray", ls="--")
ax[1].set_title("Delta & Gamma vs spot"); ax[1].set_xlabel("Spot S")
ax[1].set_ylabel("Greek value"); ax[1].legend()
plt.tight_layout(); plt.show()


<a id="nn-theory"></a>
## 5. Part II — Neural-network trading: theory

### 5.1 Idea

Classical Black–Scholes assumes **constant, known volatility** and prices
derivatives; it does **not** tell you *which direction* the market will move.
Here we take the complementary view: we keep the **volatility/return machinery of
the BS world** and let a **neural network learn a directional decision** from data.

The pipeline is:

$$
\underbrace{\text{market data}}_{\text{prices}}
\;\rightarrow\;
\underbrace{\text{BS / volatility features}}_{\text{returns, }\sigma,\text{ z-scores, Greeks}}
\;\rightarrow\;
\underbrace{\text{neural network}}_{\text{classifier}}
\;\rightarrow\;
\underbrace{\text{position}}_{\text{long / flat / short}}
\;\rightarrow\;
\underbrace{\text{backtest}}_{\text{P\&L, Sharpe}}
$$

### 5.2 What the network predicts

We frame trading as a **3-class classification** of the **next day's return**:

- class **+1 (Long)**  if next-day return $> +\tau$,
- class **0  (Flat)**  if $|$next-day return$| \le \tau$,
- class **−1 (Short)** if next-day return $< -\tau$,

with a small **dead-band** $\tau$ (a fraction of daily volatility) so the model
is not forced to bet on noise. The network outputs class probabilities via a
**softmax**; the trading position is a **volatility-scaled, Delta-like mapping**
of those probabilities into $[-1, +1]$.

### 5.3 Why Black–Scholes features?

- **Realized volatility** $\sigma$ is *the* BS parameter and strongly drives
  risk-adjusted returns; we feed several horizons of it.
- **Standardized moves** $z = r_t / \sigma_t$ are exactly the argument of the
  normal distribution in the BS formula — natural, scale-free features.
- A synthetic **BS Delta** built from a rolling z-score gives a smooth, bounded
  "how far in/out of the money is momentum" signal.

### 5.4 Avoiding look-ahead bias

Financial ML is easy to get wrong. We are careful to:

- build every feature from **past** data only (rolling windows, then `shift`),
- split **chronologically** (train → validation → test, never shuffled),
- fit the scaler on the **training set only**,
- apply realistic **transaction costs** in the backtest.


<a id="data"></a>
## 6. Part II — Real historical data

We download real daily prices with `yfinance`. If the Colab runtime has no
internet access (or the download fails), we **fall back to a GBM-simulated
series** so the whole notebook still runs end-to-end.


In [ ]:
TICKER   = "SPY"          # try e.g. "AAPL", "MSFT", "^GSPC", "BTC-USD"
START    = "2010-01-01"
END      = "2024-12-31"

def load_prices(ticker, start, end):
    '''Return a DataFrame with a 'Close' column, real or simulated.'''
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df is not None and len(df) > 250:
            out = df[["Close"]].dropna().copy()
            out.attrs["source"] = f"real data ({ticker})"
            return out
        raise ValueError("empty download")
    except Exception as e:
        print(f"[warn] download failed ({e}); using GBM-simulated fallback series.")
        n = 252 * 12
        _, p = simulate_gbm(100.0, 0.07, 0.18, n / 252, n, 1, seed=SEED)
        idx = pd.bdate_range(start=start, periods=n + 1)
        out = pd.DataFrame({"Close": p[0]}, index=idx)
        out.attrs["source"] = "SIMULATED (offline fallback)"
        return out

prices = load_prices(TICKER, START, END)
SOURCE = prices.attrs.get("source", "unknown")
print("Data source:", SOURCE)
print("Rows:", len(prices), "| from", prices.index[0].date(), "to", prices.index[-1].date())
prices.tail()


In [ ]:
plt.figure()
plt.plot(prices.index, prices["Close"], lw=1.2)
plt.title(f"{TICKER} closing price  [{SOURCE}]")
plt.xlabel("Date"); plt.ylabel("Price"); plt.tight_layout(); plt.show()


<a id="features"></a>
## 7. Part II — Feature engineering (BS / volatility based)

Every feature below is computed from **past** information only. The realized
volatility is the annualized standard deviation of log-returns — the empirical
counterpart of the BS $\sigma$.


In [ ]:
def build_features(prices, vol_windows=(5, 10, 21, 63), mom_windows=(5, 10, 21, 63)):
    df = pd.DataFrame(index=prices.index)
    df["close"] = prices["Close"]
    df["log_ret"] = np.log(df["close"]).diff()

    # --- Realized (annualized) volatility over several horizons: the BS sigma ---
    for w in vol_windows:
        df[f"vol_{w}"] = df["log_ret"].rolling(w).std() * np.sqrt(252)

    # --- Momentum / trend features ---
    for w in mom_windows:
        df[f"mom_{w}"] = df["close"].pct_change(w)

    # --- Standardized daily move  z = r_t / sigma_t  (BS-normal argument) ---
    df["z_score"] = df["log_ret"] / (df["log_ret"].rolling(21).std() + 1e-9)

    # --- Distance from moving averages (moneyness-like) ---
    for w in (21, 63):
        ma = df["close"].rolling(w).mean()
        df[f"dist_ma_{w}"] = (df["close"] - ma) / ma

    # --- RSI(14): a bounded momentum oscillator ---
    delta = df["close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    df["rsi_14"] = 100 - 100 / (1 + rs)

    # --- Synthetic Black–Scholes Delta of an ATM call whose "moneyness" is
    #     driven by the trailing z-score momentum. Smooth, bounded directional
    #     signal in (0, 1); 0.5 = neutral. ---
    roll_z = df["z_score"].rolling(10).mean().fillna(0.0)
    sig = df["vol_21"].fillna(df["vol_21"].median()).clip(0.05, 1.0)
    S_syn = 100.0 * np.exp(0.02 * roll_z)          # momentum tilts the spot
    df["bs_delta"] = bs_greeks(S_syn.values, 100.0, 0.25, 0.02, sig.values, "call")["delta"]
    df["bs_gamma"] = bs_greeks(S_syn.values, 100.0, 0.25, 0.02, sig.values, "call")["gamma"]

    return df

feat = build_features(prices)
FEATURE_COLS = [c for c in feat.columns if c not in ("close", "log_ret")]
print("Features:", FEATURE_COLS)
feat[FEATURE_COLS].tail()


<a id="config"></a>
## 7b. Strategy configuration — the levers

Before labelling and training we expose the **decision levers** in one config
dictionary, so the whole strategy can be re-tuned from a single place. These are
exactly the improvements motivated by the earlier flat-equity result:

| Lever | Meaning | Why it matters |
|-------|---------|----------------|
| `horizon` | forecast the **H-day-ahead** direction (not just tomorrow) | longer horizons have a *higher signal-to-noise ratio* than 1-day noise |
| `lookback` | sequence length fed to LSTM / GRU / CNN / Transformer | how much history the sequence models see |
| `gain` | amplifies conviction → **larger positions** | fixes the "barely invested" problem (positions were ~0.1) |
| `long_bias` | structural tilt toward being invested | equities **drift up**; a long tilt usually beats symmetric long/short on indices |
| `allow_short` | if `False`, the strategy is **long-or-flat** | avoids fighting the market's upward drift |
| `vol_target` | annualized volatility target for sizing | the BS $\sigma$ entering position sizing (risk control) |
| `max_leverage` | cap on absolute exposure | how aggressive we allow the book to get |
| `cost_bps` | transaction cost per unit of turnover | keeps the backtest honest |


In [ ]:
# --- The strategy levers, all in one place ---------------------------------
CONFIG = {
    "horizon":      5,      # predict the sign of the 5-day-ahead return (multi-day)
    "lookback":     20,     # sequence length for LSTM/GRU/CNN/Transformer
    "deadband":     0.5,    # dead-band as a fraction of H-day volatility
    "gain":         4.0,    # amplify conviction  ->  larger positions
    "long_bias":    0.15,   # structural tilt toward being invested (drift up)
    "allow_short":  False,  # LONG-BIAS regime: long-or-flat, no shorting
    "vol_target":   0.15,   # annualized vol target for position sizing
    "max_leverage": 1.5,    # cap on |exposure|
    "cost_bps":     1.0,    # per-trade cost in basis points of turnover
}
CONFIG


In [ ]:
# --- Labels: sign of the H-DAY-AHEAD return with a volatility-scaled dead-band
H = CONFIG["horizon"]
fwd_ret  = np.log(feat["close"].shift(-H) / feat["close"])   # H-day forward return
next_ret = feat["log_ret"].shift(-1)                         # 1-day return (for P&L)

# Dead-band scales with sqrt(H): a 5-day move is ~sqrt(5) larger than a 1-day one
band = CONFIG["deadband"] * feat["log_ret"].rolling(21).std() * np.sqrt(H)

label = pd.Series(0, index=feat.index)          # 0 = Flat
label[fwd_ret >  band] = 1                       # 1 = Long
label[fwd_ret < -band] = -1                      # -1 = Short

feat2 = feat.copy()
feat2["label"] = label
feat2["next_ret"] = next_ret

data = feat2.dropna().copy()
print(f"Usable samples: {len(data)}  |  horizon = {H} days")
print("Class balance (share):")
print((data["label"].value_counts(normalize=True)
       .rename({1: "Long", 0: "Flat", -1: "Short"}).round(3)))


<a id="datasets"></a>
## 8. Part II — Building the datasets: tabular **and** sequences

Different model families need different input shapes:

- **Tabular** `(samples, features)` — for the **MLP** and the **classical
  baselines** (Logistic Regression, Random Forest, Gradient Boosting). Each row
  is a snapshot of today's features.
- **Sequence** `(samples, lookback, features)` — for the **LSTM, GRU, 1D-CNN and
  Transformer**, which read a rolling *window* of the last `lookback` days.

Both views share the **same target day**, the **same chronological split**, and a
scaler fit **only on the training portion** (no look-ahead).


In [ ]:
def make_datasets(frame, feature_cols, cfg, train_frac=0.70, val_frac=0.85):
    '''Return aligned tabular + sequence datasets with a chronological split.'''
    L = cfg["lookback"]
    Xraw = frame[feature_cols].values.astype("float32")   # (n, F)
    y    = (frame["label"].values + 1).astype("int64")    # {-1,0,1}->{0,1,2}
    nr   = frame["next_ret"].values.astype("float32")
    vol  = frame["vol_21"].values.astype("float32")
    dates = frame.index
    n = len(frame)

    idx = np.arange(L - 1, n)                 # valid target rows (need L-1 history)
    n_valid = len(idx)
    i_tr = int(train_frac * n_valid)
    i_val = int(val_frac * n_valid)

    # Scaler fit ONLY on rows available before the validation set starts
    split_row = idx[i_tr]
    scaler = StandardScaler().fit(Xraw[:split_row])
    Xs = scaler.transform(Xraw).astype("float32")

    # Tabular (one row per target) and sequences (window ending at the target)
    X_tab = Xs[idx]
    X_seq = np.stack([Xs[t - L + 1: t + 1] for t in idx]).astype("float32")
    yv, nrv, volv, dv = y[idx], nr[idx], vol[idx], dates[idx]

    sl = {"tr": slice(0, i_tr), "val": slice(i_tr, i_val), "te": slice(i_val, None)}
    d = {"scaler": scaler, "n_features": Xs.shape[1], "lookback": L}
    for k, s in sl.items():
        d[f"Xtab_{k}"] = X_tab[s]
        d[f"Xseq_{k}"] = X_seq[s]
        d[f"y_{k}"]    = yv[s]
    d["test_dates"]    = dv[sl["te"]]
    d["test_next_ret"] = nrv[sl["te"]]
    d["test_vol"]      = volv[sl["te"]]
    return d

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

D = make_datasets(data, FEATURE_COLS, CONFIG)

# One-hot targets (for label smoothing) + per-sample weights (class imbalance)
n_classes = 3
classes = np.unique(D["y_tr"])
cw = compute_class_weight("balanced", classes=classes, y=D["y_tr"])
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
sw_tr = np.array([class_weight[int(c)] for c in D["y_tr"]], dtype="float32")

y_tr_oh  = keras.utils.to_categorical(D["y_tr"],  n_classes)
y_val_oh = keras.utils.to_categorical(D["y_val"], n_classes)

print(f"Features: {D['n_features']} | lookback: {D['lookback']}")
print(f"Tabular  train/val/test: {len(D['Xtab_tr'])}/{len(D['Xtab_val'])}/{len(D['Xtab_te'])}")
print(f"Sequence shape (train): {D['Xseq_tr'].shape}")
print("Class weights:", {(-1,0,1)[k]: round(v,2) for k,v in class_weight.items()})


<a id="zoo"></a>
## 9. Part II — A zoo of prediction models

We compare several **prediction models** on exactly the same data and the same
backtest, so we can judge which architecture actually helps.

### Neural networks (implemented in Keras)

| Model | Idea | Strength for markets |
|-------|------|----------------------|
| **MLP** | fully-connected net on today's features | simple, fast, strong baseline |
| **LSTM** | recurrent net with gated memory | captures temporal dependence in return sequences |
| **GRU** | lighter recurrent net (fewer gates) | similar to LSTM, faster, less overfitting |
| **1D-CNN** | temporal convolutions over the window | detects local patterns (momentum bursts, reversals) |
| **Transformer** | self-attention over the window | weighs *which past days* matter most |

### Classical baselines (scikit-learn)

| Model | Idea |
|-------|------|
| **Logistic Regression** | linear probabilistic classifier — the sanity check |
| **Random Forest** | bagged decision trees, robust to noise |
| **Gradient Boosting** | boosted trees, often the best tabular learner |

A recurring lesson in quantitative finance: on noisy tabular signals, **simple
models frequently match or beat deep nets**. Including baselines keeps us honest.
All models output calibrated 3-class probabilities `[P(Short), P(Flat), P(Long)]`
that feed the *same* position-sizing rule.


In [ ]:
from tensorflow.keras import regularizers

def _compile(model):
    model.compile(
        optimizer=keras.optimizers.Adam(5e-4),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return model

def build_mlp(n_features, dropout=0.45, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(32, activation="relu", kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Dropout(dropout),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_lstm(seq_len, n_features, units=32, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.LSTM(units, dropout=dropout, kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_gru(seq_len, n_features, units=32, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.GRU(units, dropout=dropout, kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_cnn(seq_len, n_features, filters=32, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.Conv1D(filters, 3, activation="relu", padding="causal", kernel_regularizer=reg),
        layers.Conv1D(filters // 2, 3, activation="relu", padding="causal", kernel_regularizer=reg),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_transformer(seq_len, n_features, d_model=32, heads=2, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    inp = layers.Input(shape=(seq_len, n_features))
    x = layers.Dense(d_model)(inp)                       # project to d_model
    # --- self-attention block (pre-norm residual) ---
    a = layers.LayerNormalization()(x)
    a = layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads,
                                  dropout=dropout)(a, a)
    x = layers.Add()([x, a])
    # --- feed-forward block ---
    f = layers.LayerNormalization()(x)
    f = layers.Dense(d_model, activation="relu", kernel_regularizer=reg)(f)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(d_model)(f)
    x = layers.Add()([x, f])
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(16, activation="relu", kernel_regularizer=reg)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return _compile(keras.Model(inp, out))

print("Model builders ready: MLP, LSTM, GRU, CNN, Transformer.")


In [ ]:
# --- Train every neural network on its proper input shape -------------------
def unweighted_acc(model, X, y_int):
    return float((model.predict(X, verbose=0).argmax(1) == y_int).mean())

def train_keras(model, Xtr, Xval, epochs=120, batch=128):
    cbs = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=6, min_lr=1e-5),
    ]
    return model.fit(Xtr, y_tr_oh, validation_data=(Xval, y_val_oh),
                     sample_weight=sw_tr, epochs=epochs, batch_size=batch,
                     callbacks=cbs, verbose=0)

nn_specs = {
    "MLP":         (lambda: build_mlp(D["n_features"]),                       "tab"),
    "LSTM":        (lambda: build_lstm(D["lookback"], D["n_features"]),       "seq"),
    "GRU":         (lambda: build_gru(D["lookback"], D["n_features"]),        "seq"),
    "CNN":         (lambda: build_cnn(D["lookback"], D["n_features"]),        "seq"),
    "Transformer": (lambda: build_transformer(D["lookback"], D["n_features"]),"seq"),
}

results = {}          # name -> dict(proba_te, val_acc, gap, kind)
histories = {}
for name, (factory, kind) in nn_specs.items():
    Xtr = D[f"X{kind}_tr"]; Xval = D[f"X{kind}_val"]; Xte = D[f"X{kind}_te"]
    keras.utils.set_random_seed(SEED)
    model = factory()
    h = train_keras(model, Xtr, Xval)
    tr_acc = unweighted_acc(model, Xtr, D["y_tr"])
    va_acc = unweighted_acc(model, Xval, D["y_val"])
    results[name] = {
        "proba_te": model.predict(Xte, verbose=0),
        "val_acc": va_acc, "gap": tr_acc - va_acc, "kind": "NN",
    }
    histories[name] = h
    print(f"{name:12s} | epochs {len(h.history['loss']):3d} | "
          f"train acc {tr_acc:.3f} | val acc {va_acc:.3f} | gap {tr_acc-va_acc:+.3f}")


In [ ]:
# --- Classical baselines on the tabular features ---------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

def proba_3(clf, X):
    '''Return probabilities as [P(0),P(1),P(2)] regardless of clf.classes_ order.'''
    p = clf.predict_proba(X)
    out = np.zeros((len(X), 3), dtype="float32")
    for j, c in enumerate(clf.classes_):
        out[:, int(c)] = p[:, j]
    return out

classical = {
    "LogReg":       LogisticRegression(max_iter=2000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=5,
                                           min_samples_leaf=30, class_weight="balanced",
                                           random_state=SEED, n_jobs=-1),
    "GradBoost":    HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                   max_iter=300, l2_regularization=1.0,
                                                   random_state=SEED),
}

for name, clf in classical.items():
    if name == "GradBoost":
        clf.fit(D["Xtab_tr"], D["y_tr"], sample_weight=sw_tr)
    else:
        clf.fit(D["Xtab_tr"], D["y_tr"])
    tr_acc = float((clf.predict(D["Xtab_tr"]) == D["y_tr"]).mean())
    va_acc = float((clf.predict(D["Xtab_val"]) == D["y_val"]).mean())
    results[name] = {
        "proba_te": proba_3(clf, D["Xtab_te"]),
        "val_acc": va_acc, "gap": tr_acc - va_acc, "kind": "Classical",
    }
    print(f"{name:12s} | train acc {tr_acc:.3f} | val acc {va_acc:.3f} | gap {tr_acc-va_acc:+.3f}")


In [ ]:
# --- Overfitting check across all models: validation accuracy & train-val gap
names = list(results.keys())
val_accs = [results[n]["val_acc"] for n in names]
gaps     = [results[n]["gap"] for n in names]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ["#4C72B0" if results[n]["kind"] == "NN" else "#DD8452" for n in names]
ax[0].bar(names, val_accs, color=colors)
ax[0].axhline(1/3, color="gray", ls="--", label="random (1/3)")
ax[0].set_title("Validation accuracy by model"); ax[0].set_ylabel("Accuracy")
ax[0].tick_params(axis="x", rotation=45); ax[0].legend()

ax[1].bar(names, gaps, color=colors)
ax[1].axhline(0.05, color="crimson", ls="--", label="overfit warning (~0.05)")
ax[1].set_title("Train − Validation accuracy gap (overfitting)")
ax[1].set_ylabel("Gap"); ax[1].tick_params(axis="x", rotation=45); ax[1].legend()
plt.tight_layout(); plt.show()


<a id="backtest"></a>
## 10. Part III — Backtesting all models with the new levers

### From probabilities to a position

The softmax probabilities become a **continuous target exposure**:

$$
\text{conviction}_t = \big(p^{\text{long}}_t - p^{\text{short}}_t\big)\cdot g
\; + \; b,
$$

where $g$ = `gain` amplifies the signal and $b$ = `long_bias` tilts the book
toward being invested. If `allow_short` is `False` the conviction is floored at
$0$ (**long-or-flat**). We then multiply by a **volatility-target scale**
$\min(\sigma_{\text{target}}/\sigma_t,\ \text{max\_leverage})$ — the BS $\sigma$
controlling risk — apply the exposure to **tomorrow's** return, and subtract
**transaction costs** proportional to turnover.

Every model is run through the **identical** pipeline, then ranked on a single
leaderboard by **Sharpe, return and drawdown** versus Buy & Hold.


In [ ]:
def probs_to_position(proba, realized_vol, cfg):
    p_short, p_long = proba[:, 0], proba[:, 2]
    conviction = (p_long - p_short) * cfg["gain"] + cfg["long_bias"]
    low = -1.0 if cfg["allow_short"] else 0.0
    conviction = np.clip(conviction, low, 1.0)

    rv = np.where(realized_vol > 1e-6, realized_vol, np.nan)
    scale = np.clip(cfg["vol_target"] / rv, 0.0, cfg["max_leverage"])
    scale = pd.Series(scale).ffill().fillna(1.0).values

    pos = conviction * scale
    lo = -cfg["max_leverage"] if cfg["allow_short"] else 0.0
    return np.clip(pos, lo, cfg["max_leverage"])

def run_backtest(dates, next_rets, position, cost_bps):
    turnover = np.abs(np.diff(position, prepend=0.0))
    costs = turnover * (cost_bps / 1e4)
    strat = position * next_rets - costs
    df = pd.DataFrame({"position": position, "strategy": strat, "buy_hold": next_rets},
                      index=dates)
    df["equity_strategy"] = np.exp(df["strategy"].cumsum())
    df["equity_buyhold"]  = np.exp(df["buy_hold"].cumsum())
    return df

def performance_stats(returns, periods=252):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return {}
    equity = np.exp(r.cumsum())
    dd = equity / equity.cummax() - 1
    downside = r[r < 0].std() * np.sqrt(periods)
    return {
        "total":   np.exp(r.sum()) - 1,
        "ann_ret": np.exp(r.mean() * periods) - 1,
        "ann_vol": r.std() * np.sqrt(periods),
        "sharpe":  (r.mean() / (r.std() + 1e-12)) * np.sqrt(periods),
        "sortino": (r.mean() * periods) / (downside + 1e-12),
        "max_dd":  dd.min(),
        "calmar":  (np.exp(r.mean() * periods) - 1) / (abs(dd.min()) + 1e-12),
        "win":     (r > 0).mean(),
    }

print("Backtest engine ready.")


In [ ]:
# --- Run the backtest for every model and build the leaderboard ------------
rows = []
for name, info in results.items():
    pos = probs_to_position(info["proba_te"], D["test_vol"], CONFIG)
    bt_m = run_backtest(D["test_dates"], D["test_next_ret"], pos, CONFIG["cost_bps"])
    info["bt"] = bt_m
    s = performance_stats(bt_m["strategy"])
    rows.append({
        "Model": name, "Type": info["kind"],
        "Val acc": info["val_acc"], "Gap": info["gap"],
        "Total ret": s["total"], "Ann ret": s["ann_ret"],
        "Sharpe": s["sharpe"], "Max DD": s["max_dd"],
        "Calmar": s["calmar"], "Win rate": s["win"],
    })

# Buy & Hold benchmark (same test window)
any_bt = next(iter(results.values()))["bt"]
sbh = performance_stats(any_bt["buy_hold"])
rows.append({"Model": "Buy & Hold", "Type": "Benchmark", "Val acc": np.nan, "Gap": np.nan,
             "Total ret": sbh["total"], "Ann ret": sbh["ann_ret"], "Sharpe": sbh["sharpe"],
             "Max DD": sbh["max_dd"], "Calmar": sbh["calmar"], "Win rate": sbh["win"]})

board = pd.DataFrame(rows).sort_values("Sharpe", ascending=False).reset_index(drop=True)

fmt = board.copy()
for c in ["Val acc", "Gap", "Total ret", "Ann ret", "Max DD", "Win rate"]:
    fmt[c] = fmt[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
for c in ["Sharpe", "Calmar"]:
    fmt[c] = fmt[c].map(lambda v: f"{v:.2f}")
print("OUT-OF-SAMPLE LEADERBOARD (ranked by Sharpe):\n")
print(fmt.to_string(index=False))


<a id="results"></a>
## 11. Part III — Results & charts

First, all strategies on one equity chart against Buy & Hold; then a detailed
breakdown of the **best model by Sharpe**.


In [ ]:
# --- Equity curves: every model vs Buy & Hold ------------------------------
plt.figure(figsize=(13, 6))
for name, info in results.items():
    plt.plot(info["bt"].index, info["bt"]["equity_strategy"], lw=1.3, label=name)
plt.plot(any_bt.index, any_bt["equity_buyhold"], lw=2.5, color="black",
         ls="--", label="Buy & Hold")
plt.title("Out-of-sample equity curves — all models (growth of 1)")
plt.ylabel("Equity"); plt.xlabel("Date")
plt.legend(ncol=3, fontsize=9); plt.tight_layout(); plt.show()


In [ ]:
# --- Detailed panels for the BEST model by Sharpe --------------------------
best_name = board[board["Type"] != "Benchmark"].iloc[0]["Model"]
bt = results[best_name]["bt"]
print(f"Best model by Sharpe: {best_name}")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(bt.index, bt["equity_strategy"], lw=1.8, label=f"{best_name} strategy")
ax.plot(bt.index, bt["equity_buyhold"], lw=1.4, alpha=0.8, label="Buy & Hold")
ax.set_title(f"Equity curve — {best_name}"); ax.set_ylabel("Equity"); ax.legend()

ax = axes[0, 1]
eq = bt["equity_strategy"]; dd = eq / eq.cummax() - 1
ax.fill_between(bt.index, dd, 0, color="crimson", alpha=0.4)
ax.set_title("Strategy drawdown"); ax.set_ylabel("Drawdown")

ax = axes[1, 0]
ax.plot(bt.index, bt["position"], lw=1.0, color="teal")
ax.axhline(0, color="gray", lw=0.8)
lo = -CONFIG["max_leverage"] if CONFIG["allow_short"] else 0.0
ax.set_ylim(lo - 0.1, CONFIG["max_leverage"] + 0.1)
ax.set_title("Exposure over time"); ax.set_ylabel("Position")

ax = axes[1, 1]
ax.hist(bt["strategy"], bins=40, alpha=0.7, color="slateblue")
ax.axvline(bt["strategy"].mean(), color="black", ls="--",
           label=f"mean = {bt['strategy'].mean():.4f}")
ax.set_title("Distribution of daily strategy returns")
ax.set_xlabel("Daily log-return"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# --- Rolling 63-day annualized Sharpe: best model vs Buy & Hold -------------
def rolling_sharpe(x, w=63):
    r = x.rolling(w)
    return (r.mean() / (r.std() + 1e-12)) * np.sqrt(252)

plt.figure(figsize=(12, 4))
plt.plot(bt.index, rolling_sharpe(bt["strategy"]), lw=1.4, label=best_name)
plt.plot(bt.index, rolling_sharpe(bt["buy_hold"]), lw=1.2, alpha=0.7, label="Buy & Hold")
plt.axhline(0, color="gray", lw=0.8)
plt.axhline(1, color="green", ls="--", lw=0.8, label="Sharpe = 1")
plt.title(f"Rolling 63-day annualized Sharpe — {best_name} vs Buy & Hold")
plt.ylabel("Sharpe"); plt.legend(); plt.tight_layout(); plt.show()


<a id="conclusions"></a>
## 12. Conclusions & caveats

**What we built.**
- A **Black–Scholes simulation** engine (GBM paths, closed-form pricer, Greeks).
- A **feature set grounded in the BS / volatility framework** feeding a whole
  **zoo of prediction models**: MLP, LSTM, GRU, 1D-CNN and a Transformer, plus
  classical baselines (Logistic Regression, Random Forest, Gradient Boosting).
- A tunable **decision layer** (`CONFIG`) implementing the levers that fixed the
  earlier flat-equity result: a **multi-day horizon**, a **long-bias / long-only**
  regime, higher **gain** and **leverage**, and volatility-target sizing.
- A single **leaderboard + charts** ranking every model out-of-sample on Sharpe,
  return and drawdown against Buy & Hold.

**How to read the comparison.**
- The **validation-gap chart** shows which models overfit (big train−val gap).
- The **leaderboard** and **equity chart** show which models actually add value
  *after costs*. Watch for the classic result: **simple models (LogReg / boosting)
  often rival the deep nets** on noisy financial data.
- With a long-bias regime, beating Buy & Hold in a strong bull market is hard;
  the fair question is **risk-adjusted** performance (Sharpe, Calmar, drawdown).

**Honest caveats.**
- A single train/val/test split is optimistic — use **walk-forward / purged CV**.
- Edges are thin and costs bite; treat positive Sharpe as *illustrative*.
- No slippage, borrow, market impact or regime handling beyond a flat bps cost.

**Natural next steps.**
- **Ensemble** the models (average probabilities) — often more robust than any one.
- Predict **volatility** (a true BS input) and trade **options** with the Greeks.
- Add **implied volatility** / the vol surface as features.
- Replace the fixed probability-to-position map with **reinforcement learning**.

*Educational use only — not investment advice.*
